# QM640 · 05 I2 similar-ticket search and I3 knowledge graph (RQ3)
**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. About 30–40 minutes (roughly 1 compute unit). If the session stops, run all again: saved embeddings are reused.

Needs notebooks 04 and 04b to have run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
try:
    import sentence_transformers
except ImportError:
    !pip -q install sentence-transformers
EMB_MODEL = 'BAAI/bge-small-en-v1.5'       # small open sentence-embedding model (33M parameters), runs locally
def load_backend():
    import torch
    from sentence_transformers import SentenceTransformer
    assert torch.cuda.is_available(), 'Select a T4 GPU: Runtime > Change runtime type'
    m = SentenceTransformer(EMB_MODEL, device='cuda'); m.half(); m.max_seq_length = 256
    def embed(texts):
        return m.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
    cache = {}
    def topk(E, qi, cr, limit, k):
        if cache.get('id') != id(E):
            cache.clear(); torch.cuda.empty_cache()
            cache.update(id=id(E), E=torch.from_numpy(E).cuda(), cr=torch.from_numpy(cr).cuda())
        Et, crt = cache['E'], cache['cr']; out_i, out_s = [], []
        for b in range(0, len(qi), 64):
            S = (Et[torch.from_numpy(qi[b:b+64]).cuda()] @ Et.T).float()
            S[crt[None, :] >= torch.from_numpy(limit[b:b+64]).cuda()[:, None]] = -2     # only earlier issues
            s, i = S.topk(k, dim=1); i[s <= -2] = -1
            out_i.append(i.cpu().numpy()); out_s.append(s.cpu().numpy())
        return np.vstack(out_i), np.vstack(out_s)
    return m, embed, topk


## 1. Settings
Uses the test sets from notebooks 04 and 04b. Same seed and time split.

In [ ]:
import json, time, gc, datetime, numpy as np, pandas as pd
from collections import defaultdict
from sklearn.metrics import f1_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
JIRA = DATA_ROOT/'processed'/'jira'; TESTS = DATA_ROOT/'processed'/'jira_tests'; RESULTS = DATA_ROOT/'results'
EMB_DIR = DATA_ROOT/'processed'/'jira_embeddings'; EMB_DIR.mkdir(parents=True, exist_ok=True)
SEED, K_VOTE, K_CTX, TOP_K = 640, 10, 10, 50
FILES = sorted(JIRA.glob('*.parquet'))
LABELS = json.load(open(TESTS/'label_spaces.json'))
CLS = {'D1_issuetype': 'issuetype', 'D1_priority': 'priority', 'D4_resolution': 'resolution', 'D5_component': 'proj_comp'}
RET = ('D2_duplicate', 'D6_cause')
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
def coverage(conf, ok, p):
    o = np.argsort(-conf); prec = np.cumsum(ok[o])/np.arange(1, len(o)+1); idx = np.where(prec >= p)[0]
    return round(100*(idx.max()+1)/len(o), 1) if len(idx) else 0.0
def ece(conf, ok, m=10):
    e = 0
    for a, b in zip(np.linspace(0, 1, m+1)[:-1], np.linspace(0, 1, m+1)[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): e += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return round(e, 3)
def first_comp(s):
    try: c = json.loads(s)
    except Exception: return None
    return c[0] if c else None
tests = {t: pd.read_csv(TESTS/f'testset_jira_{t}.csv') for t in list(CLS) + list(RET)}
test_keys = set().union(*[set(d['key']) for d in tests.values()])
created = pd.concat([pd.to_datetime(pd.read_parquet(f, columns=['created'])['created'], errors='coerce', utc=True) for f in FILES])
q80 = created[created >= pd.Timestamp('2000-01-01', tz='UTC')].quantile(.8).value; del created; gc.collect()

## 2. Load the embedding model
I2 uses a small open sentence-embedding model (BAAI/bge-small-en-v1.5) run locally on the GPU, so there is no per-call API cost.

In [ ]:
t_start = time.time()
model, embed, topk = load_backend()
print('embedding backend ready')

## 3. Embed all issues and find similar earlier issues
Every issue (summary + first 1,200 characters of the description) is embedded once and saved to `Data/processed/jira_embeddings`, so later notebooks reuse it. For classification, neighbours come only from before the test period, so their labels were known at the time. For duplicate and cause search, any earlier issue in the same repository can be a neighbour. About 20–30 minutes on a T4.

In [ ]:
nbrs = {}       # (task, key) -> list of (neighbour key, similarity), most similar first
meta = {}       # key -> labels and links for neighbours (only what I2/I3 need)
emb_seconds = 0
for f in FILES:
    d = pd.read_parquet(f, columns=['key', 'project', 'created', 'summary', 'description', 'issuetype', 'priority', 'resolution', 'components', 'issuelinks'])
    d['created'] = pd.to_datetime(d['created'], errors='coerce', utc=True)
    d = d[d['created'] >= pd.Timestamp('2000-01-01', tz='UTC')].reset_index(drop=True)
    keys = d['key'].values; cr = d['created'].values.astype('datetime64[ns]').astype(np.int64)
    pos = {k: i for i, k in enumerate(keys)}
    ef = EMB_DIR/f'{f.stem}.npy'
    if ef.exists() and len(np.load(ef, mmap_mode='r')) == len(d):
        E = np.load(ef)                                                # reuse embeddings from an earlier run
    else:
        t0 = time.time()
        E = embed((d['summary'].fillna('') + '. ' + d['description'].fillna('').str.slice(0, 1200)).tolist())
        emb_seconds += time.time() - t0
        np.save(ef, E); pd.DataFrame({'key': keys}).to_parquet(EMB_DIR/f'{f.stem}_keys.parquet', index=False)
    print(f.name, len(d), 'issues embedded', f'{time.time()-t_start:.0f}s')
    for t, ts in tests.items():
        q = [k for k in ts['key'] if k in pos]
        if not q: continue
        qi = np.array([pos[k] for k in q])
        # classification: neighbours from before the test period (labels known); retrieval: any issue created earlier than the query
        limit = np.full(len(qi), q80) if t in CLS else cr[qi]
        idx, sim = topk(E, qi, cr, limit, TOP_K)
        for j, k in enumerate(q): nbrs[(t, k)] = [(keys[i], float(s)) for i, s in zip(idx[j], sim[j]) if i >= 0]
    need = {n for (t, k), v in nbrs.items() for n, _ in v if n in pos} | {k for k in test_keys if k in pos}
    sub = d[d['key'].isin(need)]
    for r in sub.itertuples():
        comp = first_comp(r.components)
        meta[r.key] = {'issuetype': r.issuetype, 'priority': r.priority, 'resolution': r.resolution,
                       'proj_comp': f'{r.project} | {comp}' if comp else None, 'links': json.loads(r.issuelinks), 'created': r.created.value}
    del d, E, keys, cr, pos, sub; gc.collect()
# second pass: issues one link away from the neighbours (I3 needs their dates and labels)
far = {l.get('key') for v in list(meta.values()) for l in v['links']} - set(meta)
for f in FILES:
    d = pd.read_parquet(f, columns=['key', 'project', 'created', 'issuetype', 'priority', 'resolution', 'components'], filters=[('key', 'in', list(far))])
    d['created'] = pd.to_datetime(d['created'], errors='coerce', utc=True); d = d[d['created'].notna()]
    for r in d.itertuples():
        comp = first_comp(r.components)
        meta[r.key] = {'issuetype': r.issuetype, 'priority': r.priority, 'resolution': r.resolution,
                       'proj_comp': f'{r.project} | {comp}' if comp else None, 'links': [], 'created': r.created.value}
print(len(far), 'linked issues added for the graph')
gpu_seconds = time.time() - t_start
print(f'embedding {emb_seconds/60:.1f} min; total {gpu_seconds/60:.1f} min')

## 4. I2 alone on classification (RQ3)
The answer is a similarity-weighted vote of the 10 most similar earlier issues; confidence is the winning share. Compared with I1 on the same tickets (McNemar).

In [ ]:
def vote(task, k, space):
    w = defaultdict(float)
    used = 0
    for n, s in nbrs.get((task, k), []):
        lab = meta.get(n, {}).get(CLS[task])
        if lab is None: continue                                        # neighbour without this label (e.g. no component)
        w[lab if lab in space else 'Other'] += max(s, 0); used += 1
        if used == K_VOTE: break
    if not w: return 'Other', 0.0
    tot = sum(w.values()); best = max(w, key=w.get); return best, w[best]/tot
rows, preds = [], []
for t in CLS:
    ts = tests[t].merge(pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_pred']], on='key')
    pv = [vote(t, k, set(LABELS[t])) for k in ts['key']]
    p = np.array([a for a, _ in pv]); conf = np.array([c for _, c in pv]); y = ts['label'].values
    ok, ok1, n = p == y, ts['i1_pred'].values == y, len(y)
    b, c = int((ok1 & ~ok).sum()), int((~ok1 & ok).sum())
    rows.append({'Decision': t, 'Instrument': 'I2 similar-ticket vote (k=10)', 'Test rows': n, 'Accuracy %': round(100*ok.mean(), 1),
                 '95% CI (Wilson)': wilson(int(ok.sum()), n), 'Macro-F1': round(f1_score(y, p, average='macro', zero_division=0), 3),
                 'I1 accuracy %': round(100*ok1.mean(), 1), 'McNemar p (I2 vs I1)': round(mcnemar([[0, b], [c, 0]], exact=True).pvalue, 4),
                 'Coverage % at 98% precision': coverage(conf, ok, .98), 'Coverage % at 90% precision': coverage(conf, ok, .90), 'ECE': ece(conf, ok)})
    preds.append(pd.DataFrame({'decision': t, 'key': ts['key'], 'label': y, 'i2_pred': p, 'i2_conf': conf}))
i2cls = pd.DataFrame(rows); i2cls.to_csv(RESULTS/'rq3_jira_i2_classification.csv', index=False)
pd.concat(preds).to_csv(TESTS/'i2_predictions_classification.csv', index=False); i2cls

## 5. I3 knowledge graph
Issues are nodes; Jira links (duplicate, causes, depends, related and others) are typed edges. A new ticket enters the graph through its I2 neighbours, never through its own links, because those are the answer. Links to issues created after the ticket are ignored.

In [ ]:
# For a new ticket the graph is entered through its I2 neighbours (the ticket's own links are never used: they are the answer).
REL = {'D2_duplicate': lambda t, l: 'duplicate' in t,
       'D6_cause': lambda t, l: 'caused by' in l or (('cause' in t or 'incident' in t) and l.startswith('is caused'))}
def graph_rank(task, k, hops_from=10):
    if k not in meta: return []
    qc = meta[k]['created']; score = defaultdict(float)
    for rank, (n, s) in enumerate(nbrs.get((task, k), [])):
        if n == k: continue
        score[n] += s                                                    # the neighbour itself (I2 evidence)
        if rank < hops_from:
            for l in meta.get(n, {}).get('links', []):
                m = l.get('key')
                if not m or m == k or m not in meta or meta[m]['created'] >= qc: continue
                t, lab = (l.get('type') or '').lower(), (l.get('label') or '').lower()
                score[m] += s * (1.0 if REL[task](t, lab) else 0.3)       # follow the same relation strongly, any relation weakly
    return sorted(score, key=score.get, reverse=True)
def i3_vote(task, k, space):
    w = defaultdict(float)
    for n, s in nbrs.get((task, k), [])[:K_VOTE]:
        for m, wt in [(n, 1.0)] + [(l.get('key'), 0.5) for l in meta.get(n, {}).get('links', [])]:
            lab = meta.get(m, {}).get(CLS[task])
            if lab is None or meta[m]['created'] >= q80: continue
            w[lab if lab in space else 'Other'] += max(s, 0) * wt
    if not w: return 'Other', 0.0
    tot = sum(w.values()); best = max(w, key=w.get); return best, w[best]/tot
print('graph ready:', sum(len(v['links']) for v in meta.values()), 'typed links among', len(meta), 'issues in reach')

## 6. Duplicate (D2) and cause (D6) search: I1, I2, hybrid and I3 on the same test issues

In [ ]:
def rrf(*lists, c=60):
    sc = defaultdict(float)
    for L in lists:
        for r, x in enumerate(L): sc[x] += 1/(c + r + 1)
    return sorted(sc, key=sc.get, reverse=True)
rrows, ctx = [], []
for t in RET:
    ts = tests[t].merge(pd.read_csv(TESTS/f'i1_predictions_{t}.csv'), on='key')
    ranks = {'I1 TF-IDF lexical': [], 'I2 dense embedding': [], 'I1 + I2 hybrid (rank fusion)': [], 'I3 graph over I2 neighbours': []}
    for k, ans, i1 in zip(ts['key'], ts['answers'], ts['i1_top10']):
        truth = set(json.loads(ans)); i1 = json.loads(i1); i2 = [n for n, _ in nbrs.get((t, k), []) if n != k]
        lists = {'I1 TF-IDF lexical': i1, 'I2 dense embedding': i2, 'I1 + I2 hybrid (rank fusion)': rrf(i1, i2),
                 'I3 graph over I2 neighbours': graph_rank(t, k)}
        for m, L in lists.items():
            r = next((i+1 for i, x in enumerate(L[:10]) if x in truth), None); ranks[m].append(r)
        ctx.append({'decision': t, 'key': k, 'i2_top10': json.dumps(i2[:K_CTX]), 'i3_top10': json.dumps(lists['I3 graph over I2 neighbours'][:K_CTX]),
                    'hybrid_top10': json.dumps(lists['I1 + I2 hybrid (rank fusion)'][:K_CTX])})
    n = len(ts)
    for m, R in ranks.items():
        h = lambda kk: sum(1 for r in R if r and r <= kk)
        rrows.append({'Decision': t, 'Instrument': m, 'Test rows': n, 'Recall@1 %': round(100*h(1)/n, 1), 'Recall@5 %': round(100*h(5)/n, 1),
                      'Recall@10 %': round(100*h(10)/n, 1), 'Recall@10 95% CI': wilson(h(10), n), 'MRR@10': round(float(np.mean([1/r if r else 0 for r in R])), 3)})
ret = pd.DataFrame(rrows); ret.to_csv(RESULTS/'rq3_jira_retrieval_i1_i2_i3.csv', index=False); ret

## 7. I3 on classification, and context for the language-model runs
Saves the top-10 I2 and I3 context issues for every test ticket (`context_i2_i3.csv`, `context_tickets.parquet`). Notebooks 06–08 add these to the prompt in the RQ3 *with context* runs.

In [ ]:
rows3 = []
for t in CLS:
    ts = tests[t]; pv = [i3_vote(t, k, set(LABELS[t])) for k in ts['key']]
    p = np.array([a for a, _ in pv]); conf = np.array([c for _, c in pv]); y = ts['label'].values; ok = p == y
    rows3.append({'Decision': t, 'Instrument': 'I3 graph vote (neighbours + linked issues)', 'Test rows': len(y), 'Accuracy %': round(100*ok.mean(), 1),
                  '95% CI (Wilson)': wilson(int(ok.sum()), len(y)), 'Macro-F1': round(f1_score(y, p, average='macro', zero_division=0), 3),
                  'Coverage % at 90% precision': coverage(conf, ok, .90)})
    for k in ts['key']:
        ctx.append({'decision': t, 'key': k, 'i2_top10': json.dumps([n for n, _ in nbrs.get((t, k), [])[:K_CTX]]), 'i3_top10': None, 'hybrid_top10': None})
i3cls = pd.DataFrame(rows3); i3cls.to_csv(RESULTS/'rq3_jira_i3_classification.csv', index=False)
pd.DataFrame(ctx).to_csv(TESTS/'context_i2_i3.csv', index=False)
ctx_keys = {x for c in ctx for col in ('i2_top10', 'i3_top10') if c[col] for x in json.loads(c[col])}
cols = ['repository', 'key', 'project', 'issuetype', 'priority', 'resolution', 'created', 'summary', 'description', 'components']
pd.concat([pd.read_parquet(f, columns=cols, filters=[('key', 'in', list(ctx_keys))]) for f in FILES]).to_parquet(TESTS/'context_tickets.parquet', index=False)
print(len(ctx_keys), 'context tickets saved for the RQ3 language-model runs'); i3cls

## 8. Cost and summary

In [ ]:
GPU_UNITS_PER_HOUR = 1.2      # approximate Colab rate for a T4; check "View resources" for the actual usage
cost = pd.DataFrame([{'Instrument': 'I2 embeddings (all issues, one-off)', 'GPU minutes': round(emb_seconds/60, 1),
                      'Approx. compute units': round(emb_seconds/3600*GPU_UNITS_PER_HOUR, 2), 'Issues in reach of the graph': len(meta)}])
cost.to_csv(RESULTS/'rq3_i2_cost.csv', index=False)
def mdt(d):
    try: return d.to_markdown(index=False)
    except Exception: return d.to_string()
lines = [f'# Step 05 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', f'Embedding model: {EMB_MODEL}', '',
         '## I2 alone on classification', mdt(i2cls), '', '## I3 on classification', mdt(i3cls), '', '## Retrieval (D2, D6)', mdt(ret), '', '## Cost', mdt(cost)]
(RESULTS/'step05_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines))